In [1]:
import torch

In [2]:
X = torch.tensor([
    [1.0, 1.0],
    [1.0, 2.0],
    [2.0, 1.0],
    [2.0, 2.0],
    [8.0, 8.0],
    [8.0, 9.0],
    [9.0, 8.0],
    [9.0, 9.0],
])

y = torch.tensor([
    0, 0, 0, 0,
    1, 1, 1, 1
])

In [3]:
# 2개 특성을 가진 8개의 데이터와 이진 타겟 데이터가 존재합니다.
print("x.shape:", X.shape)
print("y.shape:", y.shape)

x.shape: torch.Size([8, 2])
y.shape: torch.Size([8])


# TensorDataset 타입
`TensorDataset`는 데이터셋과 정답 데이터가 n개 있을 때 빠르게 원하는 위치의 데이터를 주는 타입입니다.

이는 `torch.utils.data` 모듈 안에 존재합니다.

이는 Dataset 객체 안에 getitem[n] 을 실행하면 해당 위치의 튜플, `(X_tensor[n], y_tensor[n])`을 반환해줍니다.

In [6]:

from torch.utils.data import TensorDataset

dataset = TensorDataset(X, y)

print(type(dataset))
print(len(dataset))
print(dataset[0:3])
print(dataset[4])

<class 'torch.utils.data.dataset.TensorDataset'>
8
(tensor([[1., 1.],
        [1., 2.],
        [2., 1.]]), tensor([0, 0, 0]))
(tensor([8., 8.]), tensor(1))


# DataLodaer
`TensorDataset` 타입은 데이터를 묶어서 꺼내는 기능만을 제공해주게 됩니다.

하지만 여러 데이터를 묶음 단위로 사용하기에는 일일히 지정하는 것에 불편함이 존재합니다.

이를 해결하기 위해 `PyTorch`에서는 `DataLoader`이라는 데이터를 묶어주는 클래스를 제공해줍니다.

생성자의 인자로는 나누는 데이터의 크기를 나타내는 `batch_size`, 데이터를 섞은 후 분할할지 결정하는 `shuffle` 속성이 존재합니다.

`DataLoader`은 `TensorDataset`과 다르게 `Itorator` 형태이기 때문에 `for`문을 이용해서 사용할 수 있습니다.

해당 클래스 또한 `torch.utils.data`에 존재합니다.

In [7]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset=dataset,
    batch_size=3,
    shuffle=False
)

In [8]:
print(type(loader))
# 총 3개의 batch [3개짜리, 3개짜리, 2개짜리]
print(len(loader))

<class 'torch.utils.data.dataloader.DataLoader'>
3


In [15]:
for X, y in loader:
    print(" --- [X] ---")
    print(X)
    print(" --- [y] --- ")
    print(y)
    print()

 --- [X] ---
tensor([[1., 1.],
        [1., 2.],
        [2., 1.]])
 --- [y] --- 
tensor([0, 0, 0])

 --- [X] ---
tensor([[2., 2.],
        [8., 8.],
        [8., 9.]])
 --- [y] --- 
tensor([0, 1, 1])

 --- [X] ---
tensor([[9., 8.],
        [9., 9.]])
 --- [y] --- 
tensor([1, 1])



# Batch와 epoch
위와 같이 데이터를 `TensorDataset`로 나눈 뒤 `DataLodaer`을 통해 `batch`로 나누게 되면 `batch`의 개수는 `ceil(data_count / batch)`의 형태를 가지게 됩니다.

그리고 모델은 `batch` 데이터셋들을 예측 후 `loss`를 계산, `backward` 후 `step`를 통해 한 사이클을 돌게 됩니다.

이렇게 모든 `batch`를 한번 이용하면 이것을 `1_epoch`라고 부르게 됩니다.

# Batch와 epoch, 그리고 모델을 사용하여 MLP 만들기
> 이번에는 `feature` 2개짜리의 간단한 입력데이터 집합을 이용하여 최종적으로 `class 0`과 `class 1`에 대한 `logit`를 반환하는 Multilayer Perceptron을 만들어보겠습니다.

In [21]:
from torch import nn

# 새로운 예제 데이터
X = torch.tensor([
    [1., 1.],
    [1., 2.],
    [2., 1.],
    [2., 2.],
    [3., 2.],
    [7., 8.],
    [8., 7.],
    [8., 8.],
    [9., 8.],
    [9., 9.]
])

y = torch.tensor([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1
])

# 모듈 구현하기
간단하게 다음과 같은 과정을 가지는 모델을 만들어줍니다.
1. 선형 모델 (2, 4)
2. ReLU 활성화 함수
3. 선형 모델 (4, 2)

In [40]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        # (n, 2) -> (2, 4) -> ReLU -> (4, 2)
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 2)

    def forward(self, x):
        """nn.Module클래스는 __call__()을 호출 시 forward()를 호출합니다. 이에 따라 model(X)를 해도 model.forward(X)를 내부적으로 실행하게 됩니다."""
        # print("input:", x.shape)
        x = self.fc1(x)
        # print("fc1  :", x.shape)
        x = self.relu(x)
        # print("relu :", x.shape)
        x = self.fc2(x)
        # print("fc2  :", x.shape)

        return x

In [42]:
model = MLP()

In [43]:
# batch 데이터들을 준비해주기
dataset = TensorDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [44]:
for number, (X_batch, y_batch) in enumerate(loader):
    print(f"    [batch - {number}]")

    y_pred = model(X_batch)

    print(number, ":", y_pred.shape)
    print(y_pred)
    print()

    [batch - 0]
0 : torch.Size([4, 2])
tensor([[ 0.2336, -0.8498],
        [-0.3130, -0.3194],
        [ 0.2222, -0.8388],
        [ 0.2109, -0.8278]], grad_fn=<AddmmBackward0>)

    [batch - 1]
1 : torch.Size([4, 2])
tensor([[-0.2463, -0.3841],
        [ 0.3003, -0.9146],
        [-0.3244, -0.3084],
        [-0.2349, -0.3952]], grad_fn=<AddmmBackward0>)

    [batch - 2]
2 : torch.Size([2, 2])
tensor([[-0.2576, -0.3731],
        [ 0.1555, -0.7740]], grad_fn=<AddmmBackward0>)



In [45]:
for name, module in model.named_modules():
    print(name if bool(name) else "unnamed" , "->", module)

unnamed -> MLP(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=4, out_features=2, bias=True)
)
fc1 -> Linear(in_features=2, out_features=4, bias=True)
relu -> ReLU()
fc2 -> Linear(in_features=4, out_features=2, bias=True)


# Cross Entropy Loss
`CrossEntropyLoss`는 모델이 만든 `logit`에 대해서 정답 클래스의 `logit`가 다른 클래스보다 높아지도록 모델을 학습시키는 손실함수를 말합니다.

내부적으로 `logits`를 `softmax` 함수에 넣어 확률로 바꾸게 됩니다.

예를 들어 `logits`가 `[2.0, 1.0, 0.1]`와 같다면 `softmax` 함수를 이용하여 `[0.659, 0.242, 0.099]` 형태로 변환하게 됩니다.

이후에 실제 정답 클래스가 $y$라고 한다면 $L = -log(p_y)$를 손실함수로 사용하여 모델이 정답을 예측한 확률이 1에 가까울수록 `Loss`를 줄이는 방식의 손실함수입니다.

In [46]:
# 자원 준비하기
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [47]:
# epoch를 몇번 돌지 정하기
epochs = 500
loss = None

for epoch in range(1, epochs+1):
    # 학습시키기
    for X_batch, y_batch in loader:
        # 예측하기
        logits = model(X_batch)

        # 손실값 계산 + 그래프화
        loss = loss_fn(logits, y_batch)

        # 기존 기울기 초기화
        optimizer.zero_grad()

        # 역전파
        loss.backward()

        # 학습
        optimizer.step()

    # 중간 확인
    if epoch % 100 == 0:
        print(f"""
epoch: {epoch}
loss : {loss.item():.4f}
""")

X_val = torch.tensor([
# 학습 결과
    [1.5, 1.5],
    [2.5, 1.5],
    [3.0, 3.0],
    [7.0, 7.5],
    [8.5, 7.5],
    [9.5, 9.0]
], dtype=torch.float32)

y_val = torch.tensor([
    0,
    0,
    0,
    1,
    1,
    1
], dtype=torch.long)

# Module을 상속한 MLP객체에 존재하는메서드인
# .eval()을 통해서 training=False로 바꾸어 모델속 Layer들의 동작 방식을 평가용으로 변경합니다.
# Dropout라는 일부 값을 랜덤하게 0을 만드는 기능을 꺼놓거나
# BatchNorm같이 학습중 저장해둔 running mean/variance를 사용하며 업데이트하지 않습니다.
model.eval()
print("model.training:", model.training)

# torch.inference_mode()는 역전파를 위한 계산 그래프를 기록하지 말라는 것을 의미합니다.
#
with torch.inference_mode():
    logits = model(X_val)
    pred = logits.argmax(dim=1)

    accuracy = (pred == y_val).float().mean()

print("logits:")
print(logits)

print("pred:")
print(pred)

print("target:")
print(y_val)

print("accuracy:", accuracy.item())


epoch: 100
loss : 0.2726


epoch: 200
loss : 0.1078


epoch: 300
loss : 0.0670


epoch: 400
loss : 0.0566


epoch: 500
loss : 0.0418

logits:
tensor([[ 0.9474, -1.5805],
        [ 0.9474, -1.5805],
        [ 0.2938, -0.9241],
        [-2.3985,  1.7794],
        [-2.5072,  1.8885],
        [-3.3804,  2.7654]])
pred:
tensor([0, 0, 0, 1, 1, 1])
target:
tensor([0, 0, 0, 1, 1, 1])
accuracy: 1.0
